# Figure2b peptide correlation


In [ ]:
import os
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import sparse

H5AD_PATH = "data/adata_cohort1.h5ad"
GENE_SUMMARY_CSV = "results/permutation_full/sle_vs_hc/slehc_gene_level_clean_summary.csv"

ZSCORE_LAYER = "zscores_4andhalf"

QVAL_COL = "fisher_qval"
QVAL_THRESHOLD = 0.1
SORT_COL = "z_score"
TOP_K = 50

OUTPUT_DIR = "results/fig/best_peptide_correlation"
FONT_SIZE = 14
DENDRO_GAP = 0.005

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("Peptide-Peptide Correlation: Best Peptides from Top Genes")
print("=" * 70)

print("\n[1] Loading gene-level summary...")
gene_df = pd.read_csv(GENE_SUMMARY_CSV)
print(f"    Total genes: {len(gene_df):,}")

sig_df = gene_df[gene_df[QVAL_COL] <= QVAL_THRESHOLD].copy()
print(f"    Genes with {QVAL_COL} <= {QVAL_THRESHOLD}: {len(sig_df):,}")

sig_df = sig_df.sort_values(SORT_COL, ascending=False)
top_df = sig_df.head(TOP_K).copy()
n_selected = len(top_df)
print(f"    Selected top {n_selected} by {SORT_COL} (descending)")
print(f"    z_score range: {top_df[SORT_COL].min():.3f} – {top_df[SORT_COL].max():.3f}")
print(f"    log(OR) range: {top_df['log_or'].min():.3f} – {top_df['log_or'].max():.3f}")

print(f"\n    Top 10 selected genes:")
preview_cols = ["gene", "best_pep_short", "z_score", "log_or",
                "fisher_perm_p", QVAL_COL]
preview_cols = [c for c in preview_cols if c in top_df.columns]
print(top_df[preview_cols].head(10).to_string(index=False))

print(f"\n[2] Loading AnnData...")
adata = ad.read_h5ad(H5AD_PATH)
print(f"    Shape: {adata.shape[0]} samples × {adata.shape[1]} peptides")

X = adata.layers[ZSCORE_LAYER]
if sparse.issparse(X):
    X = X.toarray()

adata_peptide_idx = {p: i for i, p in enumerate(adata.var_names.tolist())}

peptide_indices = []
valid_rows = []

for row_idx, row in top_df.iterrows():
    pid = row["best_peptide"]
    if pid in adata_peptide_idx:
        peptide_indices.append(adata_peptide_idx[pid])
        valid_rows.append(row_idx)
    else:
        print(f"    WARNING: {pid} ({row['gene']}) not found in AnnData")

top_df = top_df.loc[valid_rows].reset_index(drop=True)
print(f"    Matched {len(peptide_indices)} / {n_selected} peptides in AnnData")

Z = X[:, peptide_indices]
print(f"    Z-score subset shape: {Z.shape}")

del adata, X

print(f"\n[3] Computing correlations...")

pearson_corr = np.corrcoef(Z.T)
print(f"    Pearson — shape: {pearson_corr.shape}")
ut = pearson_corr[np.triu_indices(pearson_corr.shape[0], k=1)]
print(f"      mean={np.nanmean(ut):.3f}, median={np.nanmedian(ut):.3f}, "
      f"range=[{np.nanmin(ut):.3f}, {np.nanmax(ut):.3f}]")

labels = top_df["best_pep_short"].tolist()

def plot_clustermap(corr_mat, method_name, labels, output_dir):
    corr_df = pd.DataFrame(corr_mat, index=labels, columns=labels)

    g = sns.clustermap(
        corr_df,
        cmap="RdBu_r",
        vmin=-1,
        vmax=1,
        figsize=(16, 14),
        xticklabels=True,
        yticklabels=True,
        dendrogram_ratio=(0.08, 0.08),
        cbar_pos=None,
        tree_kws={"linewidths": 0.5},
    )

    g.ax_heatmap.set_aspect('equal')

    g.fig.canvas.draw()

    hm_pos = g.ax_heatmap.get_position()

    cd_pos = g.ax_col_dendrogram.get_position()
    g.ax_col_dendrogram.set_position([
        hm_pos.x0, cd_pos.y0,
        hm_pos.width, cd_pos.height
    ])

    rd_pos = g.ax_row_dendrogram.get_position()
    rd_new_x0 = hm_pos.x0 - rd_pos.width - DENDRO_GAP
    g.ax_row_dendrogram.set_position([
        rd_new_x0, hm_pos.y0,
        rd_pos.width, hm_pos.height
    ])

    cbar_ax = g.fig.add_axes([rd_new_x0, cd_pos.y0 + cd_pos.height * 0.1,
                               0.015, cd_pos.height * 0.8])
    g.fig.colorbar(g.ax_heatmap.collections[0], cax=cbar_ax)
    cbar_ax.tick_params(labelsize=9)

    g.ax_heatmap.set_xticklabels(
        g.ax_heatmap.get_xticklabels(), fontsize=FONT_SIZE, rotation=90
    )
    g.ax_heatmap.set_yticklabels(
        g.ax_heatmap.get_yticklabels(), fontsize=FONT_SIZE
    )

    title = (
        f"Top {len(labels)} SLE-Enriched Peptides "
        f"(best per gene, {QVAL_COL}<{QVAL_THRESHOLD}, ranked by log(OR)/SE)\n"
        f"{method_name} Correlation of {ZSCORE_LAYER}"
    )
    g.fig.suptitle(title, y=1.03, fontsize=12)

    fname = f"best_peptide_correlation_{method_name.lower()}.pdf"
    save_path = os.path.join(output_dir, fname)
    g.fig.savefig(save_path, dpi=150, bbox_inches="tight")
    print(f"    Saved: {save_path}")
    plt.close()

    return corr_df

print(f"\n[4] Plotting clustermaps...")
pearson_df = plot_clustermap(pearson_corr, "Pearson", labels, OUTPUT_DIR)

print(f"\n[5] Saving outputs...")

pearson_df.to_csv(os.path.join(OUTPUT_DIR, "best_peptide_correlation_pearson.csv"))

info_df = top_df[
    [c for c in ["gene", "best_pep_short", "best_peptide", "log_or",
                  "z_score", "odds_ratio", "best_peptide_pval",
                  "prop_high_case_donors", "prop_high_control_donors",
                  "fisher_perm_p", QVAL_COL]
     if c in top_df.columns]
].copy()
info_path = os.path.join(OUTPUT_DIR, "selected_peptide_info.csv")
info_df.to_csv(info_path, index=False)
print(f"    Saved: {info_path}")

print("\n" + "=" * 70)
print("Done!")
print("=" * 70)